In [82]:
# ============================================================
# TYPE CHECKING USING LEX AND YACC
# GOOGLE COLAB - COMPLETE SINGLE CELL
# ============================================================

# Install required packages
!apt-get update -qq
!apt-get install -y flex bison gcc -qq


# -------------------- typecheck.l --------------------

with open("typecheck.l", "w") as f:
    f.write(r'''
%{
#include "typecheck.tab.h"
#include <string.h>
#include <stdlib.h>
%}

%option noyywrap

%%

"int"       { return INT; }
"float"     { return FLOAT; }

[a-zA-Z_][a-zA-Z0-9_]* {
    yylval.str = strdup(yytext);
    return ID;
}

[0-9]+ {
    yylval.str = strdup(yytext);
    return NUM;
}

"="         { return '='; }
"+"         { return '+'; }
"-"         { return '-'; }
"*"         { return '*'; }
"/"         { return '/'; }
";"         { return ';'; }

[ \t\n]+    { }

.           { return yytext[0]; }

%%
''')


# -------------------- typecheck.y --------------------

with open("typecheck.y", "w") as f:
    f.write(r'''
%{
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

struct sym
{
    char name[20];
    char type[10];
};

struct sym table[50];
int n = 0;

void insert(char *name, char *type)
{
    strcpy(table[n].name, name);
    strcpy(table[n].type, type);
    n++;
}

char *typeOf(char *name)
{
    int i;

    for (i = 0; i < n; i++)
    {
        if (strcmp(table[i].name, name) == 0)
            return table[i].type;
    }

    return "undefined";
}

int yylex(void);
int yyerror(char *s);
%}

%union
{
    char *str;
}

%token <str> ID NUM
%token INT FLOAT

%type <str> expr

%left '+' '-'
%left '*' '/'

%%

program:
    stmts
    ;

stmts:
      stmts stmt
    | stmt
    ;

stmt:
      decl
    | assign
    ;

decl:
      INT ID ';'
      {
          insert($2, "int");
      }

    | FLOAT ID ';'
      {
          insert($2, "float");
      }
    ;

assign:
    ID '=' expr ';'
    {
        char *lt = typeOf($1);

        if (strcmp(lt, "undefined") == 0)
        {
            printf("Undefined variable: %s\n", $1);
        }
        else if (strcmp(lt, $3) == 0)
        {
            printf("No type mismatch in expression: %s = ...\n", $1);
        }
        else
        {
            printf("Type mismatch in assignment to %s\n", $1);
        }
    }
    ;

expr:
      ID
      {
          $$ = typeOf($1);
      }

    | NUM
      {
          $$ = "int";
      }

    | expr '+' expr
      {
          if (strcmp($1, $3) == 0)
              $$ = $1;
          else
              $$ = "mismatch";
      }

    | expr '-' expr
      {
          if (strcmp($1, $3) == 0)
              $$ = $1;
          else
              $$ = "mismatch";
      }

    | expr '*' expr
      {
          if (strcmp($1, $3) == 0)
              $$ = $1;
          else
              $$ = "mismatch";
      }

    | expr '/' expr
      {
          if (strcmp($1, $3) == 0)
              $$ = $1;
          else
              $$ = "mismatch";
      }
    ;

%%

int main()
{
    printf("Enter declarations and expressions:\n");
    yyparse();
    return 0;
}

int yyerror(char *s)
{
    printf("Syntax Error: %s\n", s);
    return 0;
}
''')


# Remove old generated files
!rm -f typecheck.tab.c typecheck.tab.h lex.yy.c typecheck


# Generate BISON and FLEX files
!bison -d typecheck.y
!flex typecheck.l


# Compile
!gcc lex.yy.c typecheck.tab.c -o typecheck -lfl


# -------------------- INPUT --------------------

with open("input.txt", "w") as f:
    f.write("""int a;
int b;
int c;
a = b * c;
""")


# -------------------- RUN --------------------

import subprocess

result = subprocess.run(
    ["./typecheck"],
    stdin=open("input.txt", "r"),
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

print(result.stdout)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Enter declarations and expressions:
No type mismatch in expression: a = ...

